# Deep Learning-Based Forecasting of Electricity Consumption
### Replication of Nature Scientific Reports (2024) 14:6489
- **Paper DOI:** [10.1038/s41598-024-56602-4](https://doi.org/10.1038/s41598-024-56602-4)
- **Authors:** Momina Qureshi, Masood Ahmad Arbab, & Sadaqat ur Rehman
- **Dataset:** `Hospital Building Dataset.xlsx` (Phoenix, USA - 8,760 hourly readings)
---
This notebook provides a complete interactive walkthrough to load the dataset, verify monthly averages (Table 2), train the LSTM model, replicate Figure 1 and Figure 4, and benchmark all optimizers (Table 4).

## 1. Imports and Setup

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Plotting styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using compute device: {device}')

## 2. Load Dataset & Verify Against Table 2 in Paper

In [ ]:
excel_path = 'Hospital Building Dataset.xlsx'
df = pd.read_excel(excel_path)
print(f'Dataset Shape: {df.shape}')
target_col = 'Electricity:Facility [kW](Hourly)'
date_col = 'Date/Time'

# Verify monthly averages against Table 2 in the paper
df['Month'] = df[date_col].astype(str).str.strip().str.split('/').str[0]
monthly_means = df.groupby('Month')[target_col].mean().to_dict()

paper_table_2 = {
    '01': 1189.23, '02': 1166.03, '03': 1190.21, '04': 1182.77,
    '05': 1163.78, '06': 1137.14, '07': 1111.31, '08': 1132.36,
    '09': 1114.51, '10': 1143.15, '11': 1191.02, '12': 1155.29,
}

verification_df = pd.DataFrame([
    {'Month': m, 'Paper Table 2 [kW]': paper_table_2[m], 'Our Dataset [kW]': round(monthly_means[m], 2), 'Match': abs(paper_table_2[m] - monthly_means[m]) < 0.05}
    for m in paper_table_2
])
verification_df

## 3. Figure 1: Full-Year 8,760-Hour Electricity Load Profile

In [ ]:
raw_values = df[target_col].values.astype(np.float32)

fig, ax = plt.subplots(figsize=(14, 4), dpi=150)
ax.plot(raw_values, color='#1f77b4', linewidth=0.8, alpha=0.9, label='Electricity Demand')
ax.set_title('Figure 1: The electricity load profile of dataset for the 8,760 h', fontsize=13, fontweight='bold', pad=12)
ax.set_xlabel('Time (Hourly Steps)', fontsize=11)
ax.set_ylabel('Electricity Consumption [kWh]', fontsize=11)
ax.set_xlim(0, len(raw_values))
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 4. Preprocessing & Sliding Window Generation
- Sequence length $p = 60$ time-steps lookback
- 90% training split ($pprox 7,884$ hours), 10% test split ($pprox 876$ hours)
- Min-Max scaling to range $[0, 1]$

In [ ]:
seq_len = 60
train_ratio = 0.90
val_ratio = 0.10

n_total = len(raw_values)
n_train_total = int(n_total * train_ratio)
train_total = raw_values[:n_train_total]
test_values = raw_values[n_train_total:]

# Internal validation split
val_size = int(len(train_total) * val_ratio)
train_values = train_total[:-val_size]
val_values = train_total[-val_size:]

# Normalization [0, 1]
scaler = MinMaxScaler(feature_range=(0, 1))
train_scaled = scaler.fit_transform(train_values.reshape(-1, 1)).flatten()
val_scaled = scaler.transform(val_values.reshape(-1, 1)).flatten()
test_scaled = scaler.transform(test_values.reshape(-1, 1)).flatten()

# Lookback for test set
test_with_lookback = np.concatenate([train_total[-seq_len:], test_values])
test_with_lookback_scaled = scaler.transform(test_with_lookback.reshape(-1, 1)).flatten()

def create_windows(series, length):
    X, y = [], []
    for i in range(len(series) - length):
        X.append(series[i:i + length])
        y.append(series[i + length])
    return np.array(X).reshape(-1, length, 1), np.array(y).reshape(-1, 1)

X_train, y_train = create_windows(train_scaled, seq_len)
X_val, y_val = create_windows(val_scaled, seq_len)
X_test, y_test = create_windows(test_with_lookback_scaled, seq_len)

print(f'Train tensor: X={X_train.shape}, y={y_train.shape}')
print(f'Val tensor:   X={X_val.shape}, y={y_val.shape}')
print(f'Test tensor:  X={X_test.shape}, y={y_test.shape}')

## 5. Model Architecture (LSTM Regression Network)

In [ ]:
class ElectricityLSTM(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers=num_layers, batch_first=True, dropout=dropout if num_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_out = lstm_out[:, -1, :]
        out = self.fc(self.dropout(last_out))
        return out

model = ElectricityLSTM(input_size=1, hidden_size=64, num_layers=2, dropout=0.2).to(device)
print(model)

## 6. Train with Adam Optimizer (Target $R^2 \approx 95\%$)

In [ ]:
# PyTorch DataLoaders
train_loader = DataLoader(list(zip(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32))), batch_size=64, shuffle=True)
val_loader = DataLoader(list(zip(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32))), batch_size=64, shuffle=False)

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
epochs = 30
best_val_loss = float('inf')
best_weights = None

train_losses, val_losses = [], []
for epoch in range(1, epochs + 1):
    model.train()
    t_loss = 0.0
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        pred = model(bx)
        loss = criterion(pred, by)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        t_loss += loss.item() * len(bx)
    t_loss /= len(train_loader.dataset)
    train_losses.append(t_loss)

    model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(device), by.to(device)
            v_loss += criterion(model(bx), by).item() * len(bx)
    v_loss /= len(val_loader.dataset)
    val_losses.append(v_loss)

    if v_loss < best_val_loss:
        best_val_loss = v_loss
        best_weights = model.state_dict().copy()

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:02d}/{epochs:02d} | Train Loss: {t_loss:.6f} | Val Loss: {v_loss:.6f}')

model.load_state_dict(best_weights)
print('\nTraining Complete. Loaded best model checkpoint.')

## 7. Evaluate Model on Test Horizon ($R^2$, MAE, RMSE, MAPE)

In [ ]:
model.eval()
with torch.no_grad():
    preds_scaled = model(torch.tensor(X_test, dtype=torch.float32).to(device)).cpu().numpy()

# Invert scaling back to original physical kW units
preds_kw = scaler.inverse_transform(preds_scaled).flatten()
actuals_kw = test_values

r2 = r2_score(actuals_kw, preds_kw)
mae = mean_absolute_error(actuals_kw, preds_kw)
rmse = np.sqrt(mean_squared_error(actuals_kw, preds_kw))
mape = np.mean(np.abs((actuals_kw - preds_kw) / actuals_kw)) * 100.0

print('=' * 45)
print('EVALUATION RESULTS (Adam Optimizer)')
print('=' * 45)
print(f'R-squared (R2):                  {r2 * 100:.2f}%')
print(f'Paper Benchmark (Adam):          95.00%')
print(f'Mean Absolute Error (MAE):       {mae:.2f} kW')
print(f'Root Mean Squared Error (RMSE):  {rmse:.2f} kW')
print(f'Mean Absolute Percentage Error:  {mape:.2f}%')
print('=' * 45)

## 8. Figure 4: Test Horizon Actual vs. Forecasted Electricity Consumption

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5), dpi=150)
ax.plot(actuals_kw, label='Actual Demand [kW]', color='#1f77b4', linewidth=1.4, alpha=0.85)
ax.plot(preds_kw, label=f'Forecasted (Adam, R2={r2*100:.1f}%)', color='#ff7f0e', linewidth=1.4, linestyle='--')
ax.set_title('Figure 4: Actual vs. Forecasted Electricity Consumption (Test Horizon)', fontsize=12, fontweight='bold', pad=10)
ax.set_xlabel('Test Horizon (Hourly Steps)', fontsize=10)
ax.set_ylabel('Electricity Consumption [kW]', fontsize=10)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 9. Comparative Results (Replicating Table 4 from the Paper)
Load the comparative benchmark table generated across all 6 optimizers.

In [ ]:
benchmark_csv = Path('artifacts/benchmark_results_table_4.csv')
if benchmark_csv.exists():
    results_df = pd.read_csv(benchmark_csv)
    print('Replicated Benchmark Table 4:')
    display(results_df)
else:
    print('Run `python run_experiments.py --all_optimizers` to generate the complete benchmark CSV.')